In [1]:
API_KEY = ""
PINECONE_API_KEY = ""

## 7.1 Ragas SDK 설치

In [2]:
!pip install ragas

  Using cached diskcache-5.6.3-py3-none-any.whl.metadata (20 kB)
  Using cached requests-2.32.3-py3-none-any.whl.metadata (4.6 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 1.4 MB/s eta 0:00:00
  Using cached multiprocess-0.70.16-py310-none-any.whl.metadata (7.2 kB)
INFO: pip is looking at multiple versions of langchain-core to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of langchain-core to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
  Using cached langchain_core-0.1.29-py3-none-any.whl.metadata (6.0 kB)
  Using cached langchain_core-0.1.28-py3-none-any.whl.metadata (6.0 kB)
  Using cached langcha

INFO: pip is still looking at multiple versions of langchain-openai to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
  Using cached langchain_openai-0.0.8-py3-none-any.whl.metadata (2.5 kB)
  Using cached langchain_openai-0.0.7-py3-none-any.whl.metadata (2.5 kB)
  Using cached langchain_openai-0.0.6-py3-none-any.whl.metadata (2.5 kB)
  Using cached dill-0.3.8-py3-none-any.whl.metadata (10 kB)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.2/187.2 kB 4.4 MB/s eta 0:00:000:00:01
Using cached diskcache-5.6.3-py3-none-any.whl (45 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 6.9 MB/s eta 0:00:00 MB/s eta 0:00:01
Using cached langchain_core-0.1.23-py3-none-any.whl (241 kB)
Using cached langsmith-0.0.87-py3-none-any.whl (55 kB)
Using cached langchain_openai-0.0.6-py3-none-any.whl (29 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 469.0/469.0 kB 6.8 MB/s eta 0:00:009 MB/s eta 0:00:01
Using cached multiprocess-0.70.16-py310-none-any.whl (134 kB)
Using cached dill-0.3.8-py3-none-any.whl (116 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.7/30.7 MB 20.1 MB/s eta 0:00:00m eta 0:00:010:00:01
Using cached requests-2.32.3-py3-none-any.whl (64 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.5/78.5 kB 3.1 MB/s eta 0:00:00
  Attempting uninstall: tqdm
    Found existing installation: tqdm 4.66.1
    Uninstalling tqdm-4.66.1:
      Successfully uninstalled tqdm-4.66

## 7.2 Ragas 평가자 모델 정의

In [3]:
from langchain_anthropic import ChatAnthropic
from ragas.llms import LangchainLLMWrapper

llm = ChatAnthropic(
    model='claude-3-5-sonnet-20241022',
    temperature=0.1, 
    api_key=API_KEY
)

evaluator_llm = LangchainLLMWrapper(llm)

/Users/user/study/llmops/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 7.3 langchain-pinecone SDK 설치

In [ ]:
!pip install langchain-pinecone

## 7.4 Ragas 평가시에 활용할 임베딩 정의

In [4]:
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_pinecone import PineconeEmbeddings

embeddings = PineconeEmbeddings(model="multilingual-e5-large", api_key=PINECONE_API_KEY)
evaluator_embeddings = LangchainEmbeddingsWrapper(embeddings)

## 7.5 Ragas Context Precision 예시

In [6]:
from ragas import SingleTurnSample
from ragas.metrics import LLMContextPrecisionWithoutReference

# 평가 지표 정의
context_precision = LLMContextPrecisionWithoutReference(llm=evaluator_llm)
# 평가 대상 샘플 데이터
sample = SingleTurnSample(
    user_input="에펠탑은 어디에 위치해 있나요?",
    response="에펠탑은 파리에 위치해 있습니다.",
    retrieved_contexts=["에펠탑은 파리에 위치합니다.", "파리는 프랑스의 수도입니다."], 
)
# 평가 지표 계산
await context_precision.single_turn_ascore(sample)

<coroutine object SingleTurnMetric.single_turn_ascore at 0x146a118c0>

## 7.6 Ragas Context Recall 예시

In [ ]:
from ragas.dataset_schema import SingleTurnSample
from ragas.metrics import LLMContextRecall

# 평가 지표 정의
sample = SingleTurnSample(
    user_input="에펠탑은 어디에 위치해 있나요?",
    response="에펠탑은 파리에 위치해 있습니다.",
    reference="에펠탑은 파리에 위치해 있습니다.", # 참조 답변
    retrieved_contexts=["에펠탑은 파리에 위치합니다.", "파리는 프랑스의 수도입니다."], 
)
# 평가 대상 샘플 데이터
context_recall = LLMContextRecall(llm=evaluator_llm)
# 평가 지표 계산
await context_recall.single_turn_ascore(sample)

## 7.7 Ragas Faithfulness 예시

In [ ]:
from ragas.dataset_schema import SingleTurnSample 
from ragas.metrics import Faithfulness

sample = SingleTurnSample(
    user_input="최초의 슈퍼볼은 언제 열렸나요?",
    response="최초의 슈퍼볼은 1967년 1월 15일에 열렸습니다.",
    retrieved_contexts=[
        "최초의 AFL-NFL 월드 챔피언십 경기는 1967년 1월 15일, 로스앤젤레스에 있는 로스앤젤레스 메모리얼 콜리세움에서 열린 미국 풋볼 경기였습니다."
    ]
)

scorer = Faithfulness(llm=evaluator_llm)
await scorer.single_turn_ascore(sample)

## 7.8 Ragas Answer Relevancy 예시

In [ ]:
from ragas import SingleTurnSample 
from ragas.metrics import ResponseRelevancy

sample = SingleTurnSample(
        user_input="최초의 슈퍼볼은 언제 열렸나요?",
        response="최초의 슈퍼볼은 1967년 1월 15일에 열렸습니다.",
        retrieved_contexts=[
            "최초의 AFL-NFL 월드 챔피언십 경기는 1967년 1월 15일, 로스앤젤레스에 있는 로스앤젤레스 메모리얼 콜리세움에서 열린 미국 풋볼 경기였습니다."
        ]
    )

scorer = ResponseRelevancy(llm=evaluator_llm, embeddings=evaluator_embeddings)
await scorer.single_turn_ascore(sample)

## 7.11 RAG 애플리케이션 체인 평가를 위한 평가자 정의

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_anthropic import ChatAnthropic
from llmops_lib.retrievers import CustomPineconeRetriever

prompt_template = ChatPromptTemplate([
    ("system", "당신은 보험 상품 관련 고객 서비스 지원 챗봇입니다. 주어진 문서를 기반으로만 답변을 생성합니다."),
    ("user", "문서: {context}\n질문: {question}")
])


llm = ChatAnthropic(
    model='claude-3-5-sonnet-20241022',
    temperature=0.1, 
    api_key=API_KEY
)


retriever = CustomPineconeRetriever.create(
    pinecone_api_key=PINECONE_API_KEY, 
    index_name="insurance", 
    namespace="insurance-namespace"
)

def format_docs(docs):
    return '\n\n'.join([d.page_content for d in docs])

chain = (
    (lambda x: {'context': format_docs(retriever.invoke(x['question'])), 'question': x['question']})
    | prompt_template
    | llm
)

##  7.12 RAG 평가자 정의

In [ ]:
from llmops_lib.evaluators import RAGEvaluator

evaluator = RAGEvaluator(chain=chain, retriever=retriever, evaluator_llm=llm)

## 7.13 RAG 평가자 결과 예시

In [ ]:
reference = """의무보험 가입대상 자동차는 다음과 같습니다:

1. 자동차관리법 제3조 규정에 의하여 등록된 자동차

2. 건설기계관리법 제3조 규정에 의하여 등록된 건설기계 중 자배법시행령 제2조에 정한 건설기계:
- 덤프트럭
- 트럭적재식 콘크리트펌프
- 타이어식 기중기
- 트럭적재식 아스팔트살포기
- 콘크리트믹서트럭
- 타이어식 굴삭기
- 특수건설기계 중 트럭지게차
- 도로보수트럭
- 노면측정장비(노면측정장치를 가진 자주식)"""

print(evaluator.evaluate(input_variables={"question": "의무보험 가입대상 자동차가 뭐야?"}, reference_output=reference))

## 7.19 inspect 동작 예시

In [ ]:
import inspect
from llmops_lib.retrievers import CustomPineconeRetriever

signiture = inspect.signature(CustomPineconeRetriever.create)
signiture.parameters

## 7.24 PDF 파일 파서

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("../dataset/자동차보험_상품요약서.pdf")
docs = loader.load()

## 7.25 Ragas 에서 사용할 LLM과 임베딩 정의

In [ ]:
from ragas.testset import TestsetGenerator

from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

from langchain_anthropic import ChatAnthropic
from langchain_pinecone import PineconeEmbeddings

# 데이터셋 생성기
generator_llm = LangchainLLMWrapper(ChatAnthropic(model="claude-3-5-sonnet-20241022", api_key=API_KEY))
# 문서 임베딩
generator_embeddings = LangchainEmbeddingsWrapper(PineconeEmbeddings(model="multilingual-e5-large", api_key=PINECONE_API_KEY))

## 7.26 TestsetGenerator 정의

In [ ]:
from ragas.testset.persona import Persona
from ragas.testset import TestsetGenerator

# 한국어 기준으로 데이터셋을 생성하기위한 페르소나 설정
personas = [
    Persona(
        name="korean cunsumer",
        role_description="한국 고객이 질문하고, 한국어로 답변받길 원하는 사람",
    ),
]

generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings, persona_list=personas)

## 7.27 지정한 질문 유형에 따른 테스트셋 생성

In [ ]:
from ragas.testset.synthesizers.single_hop.specific import (
    SingleHopSpecificQuerySynthesizer,
)

distribution = [
    (SingleHopSpecificQuerySynthesizer(llm=generator_llm), 1.0), # 전체 데이터셋의 분포 정의(100%)
]
    
dataset = generator.generate_with_langchain_docs(
    docs[:3], # 문서 내 3개 페이지 샘플링
    testset_size=5,  # 생성할 테스트셋 크기
    query_distribution=distribution
)

## 7.28 테스트셋 샘플링 출력

In [ ]:
df = dataset.to_pandas()
query, output = df.iloc[0][["user_input", "reference"]]

print(f"Query: {query}")
print(f"Output: {output}")
